In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import hashlib
import random
import time

# --- [Paramètres du Protocole et de la Simulation] ---
P_CH_PROBABILITY = 0.2          # Probabilité qu'un noeud devienne CH
CLUSTER_CAPACITY = 3            # Nombre max de noeuds par cluster
TS_THRESHOLD = 0.5              # Seuil de confiance pour l'exclusion
W1, W2 = 0.6, 0.4               # Poids pour le calcul du score de confiance

# --- [Helpers Cryptographiques et Utilitaires] ---
def log(role, id, message, indent=0):
    """Fonction d'affichage pour les logs."""
    prefix = f"[{role}_{id}]" if id is not None else f"[{role}]"
    print(" " * indent + f"{prefix} {message}")

def hash_message(message):
    """Calcule le hash SHA-256."""
    return hashlib.sha256(str(message).encode()).hexdigest()

# Placeholders pour les opérations cryptographiques
def speck_encrypt(plaintext, key): return plaintext[::-1] + key[:4]
def speck_decrypt(ciphertext, key): return ciphertext[:-4][::-1]
def sign_message(message, private_key): return hash_message(message + str(private_key))

def verify_signature(message, signature, public_key, node_keys):
    """Vérifie une signature en recherchant la clé privée correspondante (simplification pour la simulation)."""
    for _, keys in node_keys.items():
        if keys['pk'] == public_key:
            expected_hash = hash_message(message + str(keys['sk']))
            return signature == expected_hash
    return False

# --- [Classes du Protocole] ---

class SensorNode:
    def __init__(self, node_id):
        self.id = node_id
        self.private_key = random.randint(1000, 99999)
        self.public_key = random.randint(100000, 999999)
        self.energy = 100
        self.is_ch = False
        self.cluster_head = None
        self.cluster_key = None
        self.round_last_ch = -1 / (P_CH_PROBABILITY + 1e-9)
        self.sequence_number = 0
        self.valid_msg_count = 0
        self.total_msg_count = 0

    def elect_as_ch(self, current_round):
        self.is_ch = False
        if current_round - self.round_last_ch < (1 / P_CH_PROBABILITY):
            return None
        if random.random() < P_CH_PROBABILITY:
            self.is_ch = True
            self.round_last_ch = current_round
            return ClusterHead(self)
        return None

    def join_network(self, cluster_heads):
        if not self.is_ch and cluster_heads:
            ch_to_join = random.choice(cluster_heads)
            log("SN", self.id, f"Demande à rejoindre CH_{ch_to_join.id}", 4)
            ch_to_join.initiate_join_consensus(self, cluster_heads)

    def receive_join_accept(self, cluster_key, ch):
        self.cluster_key = cluster_key
        self.cluster_head = ch
        log("SN", self.id, f"Adhésion à CH_{ch.id} acceptée.", 6)

    def update_key(self, new_key):
        self.cluster_key = new_key
        log("SN", self.id, "Clé de cluster mise à jour.", 6)

    def receive_revoke(self):
        log("SN", self.id, "Révocation reçue. Déconnexion du cluster.", 6)
        self.cluster_key = None
        self.cluster_head = None

    def send_data(self):
        if not self.cluster_key or not self.cluster_head: return None
        self.sequence_number += 1
        self.total_msg_count += 1
        self.energy -= random.uniform(0.5, 1.5)
        plain = f"data_sn{self.id}_seq{self.sequence_number}"
        return {'sender': self, 'ciphertext': speck_encrypt(plain, self.cluster_key),
                'hash': hash_message(plain), 'seq': self.sequence_number, 'energy': self.energy}

class ClusterHead:
    def __init__(self, node):
        self.id = node.id
        self.private_key, self.public_key = node.private_key, node.public_key
        self.cluster_key = str(random.randint(1000, 9999))
        self.members = []
        self.trust_scores = {}
        self.recent_seq = {}
        self.excluded = []

    def initiate_join_consensus(self, node, all_chs):
        log("CH", self.id, f"Lancement du consensus pour SN_{node.id}", 4)
        votes = [ch.vote_on_join() for ch in all_chs]
        approvals = votes.count("APPROVE")
        if approvals > len(all_chs) // 2:
            self.accept_node(node)
        else:
            log("CH", self.id, f"Consensus rejeté pour SN_{node.id}", 6)

    def vote_on_join(self):
        return "APPROVE" if len(self.members) < CLUSTER_CAPACITY else "REJECT"

    def accept_node(self, node):
        self.members.append(node)
        self.trust_scores[node.id] = 1.0
        node.receive_join_accept(self.cluster_key, self)

    def receive_data(self, msg):
        node = msg['sender']
        if node.id in self.excluded: return
        if self.recent_seq.get(node.id, 0) >= msg['seq']:
            log("CH", self.id, f"Attaque par rejeu de SN_{node.id} DÉTECTÉE", 4)
            return
        self.recent_seq[node.id] = msg['seq']
        plain = speck_decrypt(msg['ciphertext'], self.cluster_key)
        if hash_message(plain) == msg['hash']:
            node.valid_msg_count += 1
            log("CH", self.id, f"Données valides de SN_{node.id}", 4)
        else:
            log("CH", self.id, f"Hash invalide de SN_{node.id}. Message altéré.", 4)
        self.update_trust_and_check_malicious(node, msg['energy'])

    def update_trust_and_check_malicious(self, node, reported_energy):
        vmr = node.valid_msg_count / node.total_msg_count if node.total_msg_count > 0 else 0
        ec = max(0, 1 - abs(node.energy - reported_energy) / 100)
        ts = W1 * vmr + W2 * ec
        self.trust_scores[node.id] = ts
        if ts < TS_THRESHOLD and node.id not in self.excluded:
            log("CH", self.id, f"Confiance basse pour SN_{node.id} ({ts:.2f}). EXCLUSION.", 4)
            self.excluded.append(node.id)
            self.members = [m for m in self.members if m.id != node.id]
            node.receive_revoke()
            self.rotate_key("Révocation de noeud")

    def rotate_key(self, reason="Périodique"):
        log("CH", self.id, f"Rotation de la clé de cluster ({reason}).", 4)
        self.cluster_key = str(random.randint(1000, 9999))
        for member in self.members:
            member.update_key(self.cluster_key)

    def send_to_bs(self, bs):
        if not self.members and not self.excluded: return
        report = f"CH_{self.id}_rapport_membres_{len(self.members)}_exclus_{len(self.excluded)}"
        signature = sign_message(report, self.private_key)
        bs.receive_data(report, signature, self.public_key)

class BaseStation:
    def __init__(self, bs_id, all_node_keys, is_backup=False):
        self.id = bs_id
        self.all_node_keys = all_node_keys
        self.active = not is_backup
        self.is_backup = is_backup

    def fail(self):
        self.active = False
        log("BS", self.id, "PANNE DÉTECTÉE. Mise hors ligne.", 0)

    def recover(self):
        if self.is_backup:
            self.active = True
            log("BS", self.id, "PRISE DE CONTRÔLE. BS de secours maintenant active.", 0)

    def receive_data(self, report, signature, ch_public_key):
        if not self.active: return
        if verify_signature(report, signature, ch_public_key, self.all_node_keys):
            log("BS", self.id, f"Rapport signé valide reçu: '{report}'", 2)
        else:
            log("BS", self.id, f"SIGNATURE INVALIDE. Rapport rejeté.", 2)

# --- [Simulation Principale] ---
print("--- [INITIALISATION DU RÉSEAU SECDCOPA+] ---")
all_nodes = [SensorNode(i) for i in range(1, 11)]
all_node_keys = {n.id: {'pk': n.public_key, 'sk': n.private_key} for n in all_nodes}

bs = BaseStation(0, all_node_keys)
backup_bs = BaseStation(99, all_node_keys, is_backup=True)
log("NET", None, f"{len(all_nodes)} noeuds déployés. BS principale {bs.id} et de secours {backup_bs.id} prêtes.", 0)

malicious_node = random.choice(all_nodes)
log("NET", None, f"Configuration: SN_{malicious_node.id} se comportera de manière malveillante.", 0)

for r in range(1, 5):
    print(f"\n{'='*15} TOUR {r} {'='*15}")
    if not bs.active:
        log("NET", None, "La BS principale est en panne. Tentative de basculement.", 2)
        backup_bs.recover()
        bs = backup_bs

    # 1. Phase d'élection des CH
    log("PHASE", r, "Élection des CH", 2)
    potential_chs = [node.elect_as_ch(r) for node in all_nodes]
    active_chs = [ch for ch in potential_chs if ch is not None]

    if not active_chs:
        log("NET", r, "Aucun CH élu. Fin du tour.", 2)
        continue
    log("NET", r, f"CHs élus: {[ch.id for ch in active_chs]}", 2)

    # 2. Phase de formation des clusters
    log("PHASE", r, "Formation des Clusters", 2)
    for node in all_nodes:
        if not node.is_ch:
            node.join_network(active_chs)

    # 3. Phase de transmission des données
    log("PHASE", r, "Transmission des Données", 2)
    for node in all_nodes:
        if not node.is_ch and node.cluster_head:
            msg = node.send_data()
            if msg:
                if node.id == malicious_node.id and r > 1:
                    msg['hash'] = "wrong_hash"
                    log("SN", node.id, ">>> Envoi de données altérées intentionnellement <<<", 4)
                for ch in active_chs:
                    if ch.id == node.cluster_head.id:
                        ch.receive_data(msg)
                        break

    # 4. Phase d'agrégation et de maintenance
    log("PHASE", r, "Agrégation et Maintenance", 2)
    for ch in active_chs:
        ch.send_to_bs(bs)
        if r % 2 == 0:
            ch.rotate_key()

    # 5. Simulation d'une panne de la BS
    if r == 2:
        bs.fail()

print(f"\n{'='*15} FIN DE LA SIMULATION {'='*15}")

--- [INITIALISATION DU RÉSEAU SECDCOPA+] ---
[NET] 10 noeuds déployés. BS principale 0 et de secours 99 prêtes.
[NET] Configuration: SN_4 se comportera de manière malveillante.

=============== TOUR 1 ===============
  [PHASE_1] Élection des CH
  [NET_1] CHs élus: [2, 4, 10]
  [PHASE_1] Formation des Clusters
    [SN_1] Demande à rejoindre CH_10
    [CH_10] Lancement du consensus pour SN_1
      [SN_1] Adhésion à CH_10 acceptée.
    [SN_3] Demande à rejoindre CH_2
    [CH_2] Lancement du consensus pour SN_3
      [SN_3] Adhésion à CH_2 acceptée.
    [SN_5] Demande à rejoindre CH_10
    [CH_10] Lancement du consensus pour SN_5
      [SN_5] Adhésion à CH_10 acceptée.
    [SN_6] Demande à rejoindre CH_10
    [CH_10] Lancement du consensus pour SN_6
      [SN_6] Adhésion à CH_10 acceptée.
    [SN_7] Demande à rejoindre CH_4
    [CH_4] Lancement du consensus pour SN_7
      [SN_7] Adhésion à CH_4 acceptée.
    [SN_8] Demande à rejoindre CH_10
    [CH_10] Lancement du consensus pour SN_8
   